In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [ ]:
### +++ LOAD DATA +++ ###

diags=['sst','nt_Heat_Atl','sf_Atl']
dat={ 'ctrl':{}, 'max.global':{} }

# number of years to omit 
omit_yrs = 30
n_omit = omit_yrs * 12
tmax = 350 * 12

for key in dat.keys():    
    for varn in diags:
        dat[key][varn] = xr.open_dataset(f'aogcm.{key}.{varn}.nc')[varn].isel(time=slice(n_omit,tmax))

        
        = da.groupby('time.year').mean(dim='time') # calculate annual mean

In [ ]:
## Load Data
# Control
dfile=f'{dpath0}/model_output/e2pt1/Control/nt_Heat_Atl_2650-2999.E2pt1_PIctrl_restart.nc'
amocPi=xr.open_dataset(f'{dfile}').nt_Heat_Atl
# MAX
dfile=f'{dpath0}/model_output/e2pt1/Maxtopo/nt_Heat_Atl_timeseries.maxtopo_E2pt1_PI_restart_new.nc'
amocMax=xr.open_dataset(f'{dfile}').nt_Heat_Atl
lato2=xr.open_dataset(f'{dfile}').lato2
time=xr.open_dataset(f'{dfile}').time

## Apply weights based on latitude to account for differences in gridcell area
weights=np.cos(np.deg2rad(lato2))
weights.name='weights'
amocPi_wgt=amocPi.weighted(weights).mean(('lato2'))
amocMax_wgt=amocMax.weighted(weights).mean(('lato2'))

## Calculate annual mean from monthly data
amocPi_wgt_ann=amocPi_wgt.groupby('time.year').mean(dim='time')
amocMax_wgt_ann=amocMax_wgt.groupby('time.year').mean(dim='time')

In [ ]:
### +++ Load Data +++ ###
n_omit = 30 * 12
t_max = 350 * 12

# Control
dfile = f'{dpath0}/model_output/e2pt1/Control/sf_Atl_2650-2999.E2pt1_PIctrl_restart.nc'
sfAtl_ctrl = xr.open_dataset(f'{dfile}').sf_Atl.isel(time=slice(n_omit,t_max)).mean(dim="time")
sfAtl_ctrl = ctrl.where(ctrl > -6.8550344e+20, np.nan)
depth = xr.open_dataset(f'{dfile}').zoce
lato = xr.open_dataset(f'{dfile}').lato2

# MAX
dfile = f'{dpath0}/model_output/e2pt1/Maxtopo/sf_Atl_timeseries.maxtopo_E2pt1_PI_restart_new.nc'
sfAtl_maxtopo = xr.open_dataset(f'{dfile}').sf_Atl.isel(time=slice(n_omit,t_max)).mean(dim="time")
sfAtl_maxtopo = maxtopo.where(maxtopo > -6.8550344e+20, np.nan)


In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# plot specs
legend_kw={'loc':0, 'fontsize':11, 'labelcolor':'linecolor', 'frameon':True}
text_kw={'weight':'bold', 'color':'k', 'ha':'left', 'va':'bottom'}
line_kw={'ls': '-', 'lw': 1}
tkw={'color': 'k', 'labelsize': 12, 'labelcolor': 'k'}
# var
tm=np.linspace(0.,len(amocPi_wgt_ann.year),len(amocPi_wgt.time))
ty=np.linspace(0.,len(amocPi_wgt_ann.year),len(amocPi_wgt_ann.year))
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-150., -55., -55., 30.]
# months to average over
season='ann'
mons=get_season(season)
# sst colormap
cmap,_,_,_=get_settings(field='sst', diff=True)
vmin=-8
vmax=8
levels=np.linspace(vmin, vmax, 17)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# sf colormap
cmap2 = plt.get_cmap('YlGnBu')
vmin2 = -10
vmax2 = 30
levels2 = np.linspace(vmin2, vmax2, 21)
norm2 = mpl.colors.BoundaryNorm(levels2, cmap2.N)

# ------------------- #
#      Make Plot      #
# ------------------- #
fig=plt.figure(figsize=(10, 15), layout='constrained') 

## SST map
ax=plt.subplot2grid((3,2), (0, 0), colspan=2, rowspan=1, projection=proj) 
ax.text(-180,91,'a. MAX$_{global}$ $-$ CTRL $\Delta$SST', size=14, **text_kw)
vari=cpld.sst[3]-cpld.sst[0]
ax.pcolormesh(lon, lat, vari[mons].mean(dim='month'), cmap=cmap, vmin=vmin, vmax=vmax, transform=trans)
# add sw diff contours
zvar=maxtopo-default
zlvls=np.linspace(500, 4500, 9)
cs=ax.contour(lon, lat, zvar, levels=zlvls, linewidths=0.75, colors='k', transform=proj, zorder=7)
# settings
ax.coastlines(color='k')
ax.set_global() 
gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
gl.xformatter=LONGITUDE_FORMATTER
gl.yformatter=LATITUDE_FORMATTER
gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':10}
gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':10}
# add colorbar
cf=mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
cax=fig.add_axes([.975, 0.71, 0.0175, 0.25])
cbar=fig.colorbar(cf, orientation='vertical', extend='both', cax=cax)
cbar.set_label('[°C]', y=1.115, labelpad=-25, rotation=0, size=12, fontweight='bold', ha='center')
cbar.ax.tick_params(labelsize=12)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')
    
## Streamfunction
ax=plt.subplot2grid((3,2), (1, 0), colspan=1, rowspan=1) 
ax.text(-35,-75,'b. ATLANTIC STREAMFUNCTION ', size=14, **text_kw)
ax.text(-33.4,4935, 'CTRL', size=18, **text_kw,
           bbox=dict(facecolor='white', edgecolor='black'))
ax.contourf(lato, depth, sfAtl_ctrl, cmap=cmap2, levels=levels2, extend='both')
ax.set(xlabel="Latitude [°N]",
       xlim=[-35, 82],
       ylabel="Depth [mbsl]",
       ylim=[5000, 0])

ax=plt.subplot2grid((3,2), (1, 1), colspan=1, rowspan=1) 
ax.text(-33.4,4935, 'MAX$_{global}$', size=18, **text_kw,
           bbox=dict(facecolor='white', edgecolor='black'))
ax.contourf(lato, depth, sfAtl_maxtopo, cmap=cmap2, levels=levels2, extend='both')
ax.set(xlabel="Latitude [°N]",
       xlim=[-35, 82],
       ylim=[5000, 0])

# add colorbar
cf=mpl.cm.ScalarMappable(norm=norm2, cmap=cmap2)
cax=fig.add_axes([1.0, 0.375, 0.0175, 0.26])
cbar=fig.colorbar(cf, orientation='vertical', extend='both', cax=cax)
cbar.set_label('[Sv]', y=1.115, labelpad=-25, rotation=0, size=12, fontweight='bold', ha='center')
cbar.ax.tick_params(labelsize=12)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')
    
    
## Heat Transport timeseries
ax=plt.subplot2grid((3,2), (2, 0), colspan=2, rowspan=1) 
ax.text(0,1,'c. ATLANTIC NORTHWARD HEAT TRANSPORT ', size=14, **text_kw)
# Control
ax.plot(tm, amocPi_wgt, c='grey', **line_kw, label='CTRL')
ax.plot(ty, amocPi_wgt_ann, c='k', **line_kw, label='_Hidden')
# Max
ax.plot(tm, amocMax_wgt, c='r', **line_kw, label='MAX$_{global}$')
ax.plot(ty, amocMax_wgt_ann, c='k', **line_kw, label='_Hidden')
# 0 line
ax.axhline(y=0, color='gray', lw=1, ls='--', label='_Hidden')
# settings
ax.set_xlabel('SIMULATION YEAR', size=12)
ax.set_ylabel('[10$^{15}$ W]', size=12)
ax.set(xlim=[0,350], ylim=[-.4,1])
ax.legend(**legend_kw)
ax.tick_params(**tkw)

plt.savefig(f'{opath}/sfAtl.heat.sst.pdf', transparent=True, bbox_inches='tight')